**Genomic Benchark - Test on random SubSamples before whole set**

In [10]:
%%capture
!pip install -U genomic-benchmarks transformers accelerate scikit-learn torch

In [13]:
import transformers
print(f'Transformers version: {transformers.__version__}')

Transformers version: 5.10.1


In [11]:
import random
import numpy as np
import torch
from genomic_benchmarks.dataset_getters.pytorch_datasets import HumanNontataPromoters
from transformers import AutoTokenizer, AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# Set seeds for reproducibility
SEED = 1908
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


In [17]:
from transformers import AutoConfig, AutoModel, AutoTokenizer

# 1. Load dataset
train_dataset = HumanNontataPromoters(split='train')
test_dataset = HumanNontataPromoters(split='test')

# 2. Setup Model with manual config fixes for 'is_decoder', 'rope_theta', etc.
model_name = 'InstaDeepAI/nucleotide-transformer-v2-50m-multi-species'

# Load config and inject missing attributes
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

# Essential fixes for newer Transformers versions
fix_attrs = {
    'rope_theta': 10000.0,
    'is_decoder': False,
    'add_cross_attention': False,
    'chunk_size_feed_forward': 0
}

for attr, val in fix_attrs.items():
    setattr(config, attr, val)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name,
    config=config,
    trust_remote_code=True,
    ignore_mismatched_sizes=True
).to(device)

model.eval()
print("Model loaded successfully with config fixes.")

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: InstaDeepAI/nucleotide-transformer-v2-50m-multi-species
Key                                              | Status     |                                                                                             
-------------------------------------------------+------------+---------------------------------------------------------------------------------------------
lm_head.layer_norm.weight                        | UNEXPECTED |                                                                                             
lm_head.bias                                     | UNEXPECTED |                                                                                             
lm_head.dense.bias                               | UNEXPECTED |                                                                                             
lm_head.decoder.weight                           | UNEXPECTED |                                                      

Model loaded successfully with config fixes.


In [21]:
from collections import Counter

def check_distribution(dataset, name):
    labels = [dataset[i][1] for i in range(len(dataset))]
    counts = Counter(labels)
    print(f"Distribution for {name}:")
    for label, count in counts.items():
        print(f"  Class {label}: {count} samples ({count/len(dataset)*100:.2f}%)")

print(f"Total Train samples: {len(train_dataset)}")
print(f"Total Test samples: {len(test_dataset)}")

# Checking full distribution
check_distribution(train_dataset, 'Train Dataset')
check_distribution(test_dataset, 'Test Dataset')

# Also check if labels are ordered (e.g., all 0s then all 1s)
print("\nFirst 10 labels:", [train_dataset[i][1] for i in range(10)])
print("Last 10 labels:", [train_dataset[len(train_dataset)-1-i][1] for i in range(10)])

Total Train samples: 27097
Total Test samples: 9034
Distribution for Train Dataset:
  Class 0: 14742 samples (54.40%)
  Class 1: 12355 samples (45.60%)
Distribution for Test Dataset:
  Class 0: 4915 samples (54.41%)
  Class 1: 4119 samples (45.59%)

First 10 labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Last 10 labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [22]:
import pandas as pd

def get_class_stats(dataset, name):
    labels = [dataset[i][1] for i in range(len(dataset))]
    df = pd.Series(labels)
    counts = df.value_counts().sort_index()
    percentages = df.value_counts(normalize=True).sort_index() * 100

    stats = pd.DataFrame({
        'Count': counts,
        'Percentage (%)': percentages
    })
    print(f'\n--- {name} ---')
    display(stats)

get_class_stats(train_dataset, 'Training Dataset')
get_class_stats(test_dataset, 'Test Dataset')


--- Training Dataset ---


,Count,Percentage (%)
0,14742,54.404547
1,12355,45.595453



--- Test Dataset ---


,Count,Percentage (%)
0,4915,54.405579
1,4119,45.594421


In [18]:
from transformers import AutoConfig, AutoModel, AutoTokenizer

# Robust config fix for Nucleotide Transformer v2
model_name = "InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

# Manually set attributes that are sometimes missing in specific transformers versions
missing_attrs = {
    "rope_theta": 10000.0,
    "is_decoder": False,
    "add_cross_attention": False,
    "chunk_size_feed_forward": 0
}

for attr, value in missing_attrs.items():
    if not hasattr(config, attr):
        setattr(config, attr, value)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
# Added ignore_mismatched_sizes=True to handle the EsmModel load report mismatch
model = AutoModel.from_pretrained(model_name, config=config, trust_remote_code=True, ignore_mismatched_sizes=True).to(device)
model.eval()

Loading weights:   0%|          | 0/174 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: InstaDeepAI/nucleotide-transformer-v2-50m-multi-species
Key                                              | Status     |                                                                                             
-------------------------------------------------+------------+---------------------------------------------------------------------------------------------
lm_head.layer_norm.weight                        | UNEXPECTED |                                                                                             
lm_head.bias                                     | UNEXPECTED |                                                                                             
lm_head.dense.bias                               | UNEXPECTED |                                                                                             
lm_head.decoder.weight                           | UNEXPECTED |                                                      

EsmModel(
  (embeddings): EsmEmbeddings(
    (word_embeddings): Embedding(4107, 512, padding_idx=1)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (rotary_embeddings): EsmRotaryEmbedding()
  (encoder): EsmEncoder(
    (layer): ModuleList(
      (0-11): 12 x EsmLayer(
        (attention): EsmAttention(
          (self): EsmSelfAttention(
            (query): Linear(in_features=512, out_features=512, bias=True)
            (key): Linear(in_features=512, out_features=512, bias=True)
            (value): Linear(in_features=512, out_features=512, bias=True)
          )
          (output): EsmSelfOutput(
            (dense): Linear(in_features=512, out_features=512, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (LayerNorm): LayerNorm((512,), eps=1e-12, elementwise_affine=True)
        )
        (intermediate): EsmIntermediate(
          (dense): Linear(in_features=512, out_features=2048, bias=True)
        )
        (output): EsmOutput(
          (

In [23]:
def get_embeddings(dataset, num_samples, shuffle=True):
    # Get all indices and shuffle them to ensure multiple classes are present
    indices = list(range(len(dataset)))
    if shuffle:
        random.seed(SEED)
        random.shuffle(indices)

    # Select the requested number of samples
    selected_indices = indices[:num_samples]

    embeddings_list = []
    labels_list = []
    batch_size = 8

    with torch.no_grad():
        for i in range(0, len(selected_indices), batch_size):
            batch_idx = selected_indices[i:i+batch_size]
            batch_data = [dataset[idx] for idx in batch_idx]
            sequences = [item[0] for item in batch_data]
            labels = [item[1] for item in batch_data]

            inputs = tokenizer(sequences, return_tensors="pt", padding=True, truncation=True).to(device)
            outputs = model(**inputs)

            # Mean pooling over tokens (dimension 1)
            embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
            embeddings_list.append(embeddings)
            labels_list.extend(labels)

    return np.vstack(embeddings_list), np.array(labels_list)

# Test on shuffled subset: 1000 training, 500 test
print("Extracting shuffled embeddings for subset...")
X_train, y_train = get_embeddings(train_dataset, 1000)
X_test, y_test = get_embeddings(test_dataset, 500)

print(f"Unique classes in train: {np.unique(y_train)}")
print(f"Unique classes in test: {np.unique(y_test)}")

Extracting shuffled embeddings for subset...
Unique classes in train: [0 1]
Unique classes in test: [0 1]


In [24]:
clf = LogisticRegression(max_iter=1000, random_state=SEED)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred):.4f}')
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))
print('\nClassification Report:')
print(classification_report(y_test, y_pred))

Accuracy: 0.6400
F1 Score: 0.6104

Confusion Matrix:
[[179  87]
 [ 93 141]]

Classification Report:
              precision    recall  f1-score   support

           0       0.66      0.67      0.67       266
           1       0.62      0.60      0.61       234

    accuracy                           0.64       500
   macro avg       0.64      0.64      0.64       500
weighted avg       0.64      0.64      0.64       500



### Skalierung auf den vollen Datensatz
Hier berechnen wir die Embeddings für alle verfügbaren Daten und trainieren den Klassifikator erneut.

In [26]:
# 1. Embeddings für den gesamten Datensatz extrahieren
print(f"Extrahiere Embeddings für den gesamten Trainingsdatensatz ({len(train_dataset)} Samples)...")
X_train_full, y_train_full = get_embeddings(train_dataset, num_samples=len(train_dataset))

print(f"Extrahiere Embeddings für den gesamten Testdatensatz ({len(test_dataset)} Samples)...")
X_test_full, y_test_full = get_embeddings(test_dataset, num_samples=len(test_dataset))

# 2. Modell auf vollen Daten trainieren
clf_full = LogisticRegression(max_iter=1000, random_state=SEED)
clf_full.fit(X_train_full, y_train_full)

# 3. Evaluation
y_pred_full = clf_full.predict(X_test_full)

print(f'Full Accuracy: {accuracy_score(y_test_full, y_pred_full):.4f}')
print(f'Full F1 Score: {f1_score(y_test_full, y_pred_full):.4f}')
print('\nClassification Report (Full):')
print(classification_report(y_test_full, y_pred_full))

Extrahiere Embeddings für den gesamten Trainingsdatensatz (27097 Samples)...
Extrahiere Embeddings für den gesamten Testdatensatz (9034 Samples)...
Full Accuracy: 0.7192
Full F1 Score: 0.6912

Classification Report (Full):
              precision    recall  f1-score   support

           0       0.74      0.74      0.74      4915
           1       0.69      0.69      0.69      4119

    accuracy                           0.72      9034
   macro avg       0.72      0.72      0.72      9034
weighted avg       0.72      0.72      0.72      9034

